In [1]:
import numpy as np
import pandas as pd
import pickle
import sys
sys.path.append("/home/ethan2/GrowthNet/sweeps") 
import data_class
import matplotlib.pyplot as plt

In [2]:
binders = {
    "dihydrofolate_reductase": {
        "dihydrofolate_DHF": "[H][C@@](CCC(O)=O)(NC(=O)C1=CC=C(NCC2=NC3=C(NC2)NC(=N)N=C3O)C=C1)C(O)=O",
        "tetrahydrofolate_THF": "C1C(NC2=C(N1)N=C(NC2=O)N)CNC3=CC=C(C=C3)C(=O)N[C@@H](CCC(=O)O)C(=O)O",
        "NADPH": "OC1C(O)C(OC1N1C=CCC(=C1)C(=O)N)COP(=O)(OP(=O)(OCC1OC(C(C1O)OP(=O)(O)O)n1cnc2c1ncnc2N)O)O",
        "trimethoprim": "COc1cc(Cc2cnc(nc2N)N)cc(c1OC)OC",
        "methotrexate": "CN(C)c1nc(N)c2nc(nc2n1)N(CC1=CN(C)C(=O)N=C1N)C1=CC=C(C=C1)C(=O)N[C@@H](CCC(O)=O)C(O)=O"
    },
    "dihydrofolate_reductase_type_2": {
        "dihydrofolate_DHF": "[H][C@@](CCC(O)=O)(NC(=O)C1=CC=C(NCC2=NC3=C(NC2)NC(=N)N=C3O)C=C1)C(O)=O",
        "tetrahydrofolate_THF": "C1C(NC2=C(N1)N=C(NC2=O)N)CNC3=CC=C(C=C3)C(=O)N[C@@H](CCC(=O)O)C(=O)O",
        "NADPH": "OC1C(O)C(OC1N1C=CCC(=C1)C(=O)N)COP(=O)(OP(=O)(OCC1OC(C(C1O)OP(=O)(O)O)n1cnc2c1ncnc2N)O)O",
        "trimethoprim": "COc1cc(Cc2cnc(nc2N)N)cc(c1OC)OC"
    }
}
all_cmp_meta_path='/home/ethan2/GrowthNet/data/splits/Celine_v1/all_compound_metas.pkl'

In [3]:
from sweeps.data_class import *
from rdkit import Chem

with open(all_cmp_meta_path, "rb") as f:
    all_metas = pickle.load(f)

print(f"Loaded {len(all_metas)} compound metas")
print(f"Type: {type(all_metas[0])}")

# Get SMILES from train data
train_smiles = set()
for meta in all_metas:
    if hasattr(meta, 'smiles'):
        train_smiles.add(meta.smiles)

print(f"\nTrain set SMILES: {len(train_smiles)} unique")

Loaded 34700 compound metas
Type: <class 'data_class.CompoundMeta'>

Train set SMILES: 34681 unique


In [4]:
# Canonicalize binder SMILES
canonical_binder_smiles = {}
for target, compounds in binders.items():
    for name, smi in compounds.items():
        mol = Chem.MolFromSmiles(smi)
        if mol:
            canonical = Chem.MolToSmiles(mol)
            canonical_binder_smiles[canonical] = f"{target}:{name}"
        else:
            print(f"Failed to parse: {target}:{name}")

print(f"Binder SMILES (canonical): {len(canonical_binder_smiles)} unique")

Failed to parse: dihydrofolate_reductase:methotrexate
Binder SMILES (canonical): 4 unique


[11:43:32] Can't kekulize mol.  Unkekulized atoms: 3 4 5 7 8 9 10 11 12


In [5]:
# Check which binders are in train set
print("BINDERS IN TRAIN SET:")
print("-" * 60)
in_train = []
not_in_train = []

for canonical_smi, source in canonical_binder_smiles.items():
    if canonical_smi in train_smiles:
        in_train.append((source, canonical_smi))
        print(f"✓ {source}")
    else:
        not_in_train.append((source, canonical_smi))

print(f"\nBINDERS NOT IN TRAIN SET:")
print("-" * 60)
for source, canonical_smi in not_in_train:
    print(f"✗ {source}")

print(f"\n\nSUMMARY:")
print(f"  In train set: {len(in_train)}/{len(canonical_binder_smiles)}")
print(f"  Not in train set: {len(not_in_train)}/{len(canonical_binder_smiles)}")

BINDERS IN TRAIN SET:
------------------------------------------------------------
✓ dihydrofolate_reductase_type_2:trimethoprim

BINDERS NOT IN TRAIN SET:
------------------------------------------------------------
✗ dihydrofolate_reductase_type_2:dihydrofolate_DHF
✗ dihydrofolate_reductase_type_2:tetrahydrofolate_THF
✗ dihydrofolate_reductase_type_2:NADPH


SUMMARY:
  In train set: 1/4
  Not in train set: 3/4


# Train/Val Similarity Analysis

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

train_smiles_path = '/home/ethan2/GrowthNet/data/splits/Celine_v1/train_smiles.txt'
val_smiles_path   = '/home/ethan2/GrowthNet/data/splits/Celine_v1/val_smiles.txt'

with open(train_smiles_path) as f:
    train_set = set(f.read().splitlines())
with open(val_smiles_path) as f:
    val_set = set(f.read().splitlines())

train_metas = [m for m in all_metas if m.smiles in train_set]
val_metas   = [m for m in all_metas if m.smiles in val_set]
print(f"Train: {len(train_metas)}, Val: {len(val_metas)}, Total: {len(all_metas)}")

Train: 29774, Val: 4926, Total: 34700


In [13]:
sum(meta.is_active_at_12_50 for meta in all_metas)   

456

In [26]:
all_metas[0].pivot_cls

Concentration,0.2,1.2,7.9,50.0
Timepoint,,,,
0.00,0,0,0,0
2.08,0,0,0,0
4.16,0,0,0,0
6.24,0,0,0,0
8.32,0,0,0,0
10.40,0,0,0,0
12.48,0,0,0,0


In [14]:
len([meta for meta in all_metas if meta.is_active_at_12_50]) 

456

In [24]:
count = 0                                                              
for meta in all_metas:                                                 
    # Check if there's at least one 1 in the pivot_cls DataFrame
    if (meta.pivot_cls == 1).any().any():                              
        count += 1
                                                                        
print(f"Number of CompoundMeta objects with at least 1 active pair: {count}")

Number of CompoundMeta objects with at least 1 active pair: 741


In [9]:
# Build concatenated fingerprint matrices (MACCS 167 + ECFP 2048 + RDKit 2048 = 4263 dims)
def get_fp(meta):
    return np.concatenate([
        meta.fps_by_family['maccs_fp'],
        meta.fps_by_family['ecfp_fp'],
        meta.fps_by_family['rdkit_fp'],
    ])

train_fps = np.array([get_fp(m) for m in train_metas], dtype=np.float32)
val_fps   = np.array([get_fp(m) for m in val_metas],   dtype=np.float32)
print(f"Train FP matrix: {train_fps.shape}, Val FP matrix: {val_fps.shape}")

Train FP matrix: (29774, 4263), Val FP matrix: (4926, 4263)


In [ ]:
# Cosine similarity (batched to avoid OOM on ~29k x 5k matrix)
BATCH = 500
n_train = len(train_fps)
cosine_mean = np.zeros(n_train, dtype=np.float32)
cosine_max  = np.zeros(n_train, dtype=np.float32)

for start in range(0, n_train, BATCH):
    end = min(start + BATCH, n_train)
    sim = cosine_similarity(train_fps[start:end], val_fps)  # (batch, n_val)
    cosine_mean[start:end] = sim.mean(axis=1)
    cosine_max[start:end]  = sim.max(axis=1)

print(f"Cosine mean: {cosine_mean.mean():.4f} ± {cosine_mean.std():.4f}")
print(f"Cosine max:  {cosine_max.mean():.4f} ± {cosine_max.std():.4f}")

In [ ]:
# Tanimoto similarity: T(A,B) = (A·B) / (||A||² + ||B||² - A·B)
# Valid for binary fingerprints stored as float32
train_norm2 = (train_fps ** 2).sum(axis=1, keepdims=True)  # (n_train, 1)
val_norm2   = (val_fps   ** 2).sum(axis=1)                  # (n_val,)

tanimoto_mean = np.zeros(n_train, dtype=np.float32)
tanimoto_max  = np.zeros(n_train, dtype=np.float32)

for start in range(0, n_train, BATCH):
    end = min(start + BATCH, n_train)
    dot   = train_fps[start:end] @ val_fps.T                         # (batch, n_val)
    denom = train_norm2[start:end] + val_norm2 - dot                 # (batch, n_val)
    tan   = dot / denom.clip(min=1e-10)
    tanimoto_mean[start:end] = tan.mean(axis=1)
    tanimoto_max[start:end]  = tan.max(axis=1)

print(f"Tanimoto mean: {tanimoto_mean.mean():.4f} ± {tanimoto_mean.std():.4f}")
print(f"Tanimoto max:  {tanimoto_max.mean():.4f} ± {tanimoto_max.std():.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cosine plots
axes[0, 0].hist(cosine_mean, bins=100, color='steelblue', edgecolor='none')
axes[0, 0].axvline(np.median(cosine_mean), color='k', linestyle='--', label=f'median={np.median(cosine_mean):.3f}')
axes[0, 0].set_title("Cosine — Mean similarity to val set")
axes[0, 0].set_xlabel("Avg cosine similarity")
axes[0, 0].legend()

axes[0, 1].hist(cosine_max, bins=100, color='steelblue', edgecolor='none')
axes[0, 1].axvline(np.median(cosine_max), color='k', linestyle='--', label=f'median={np.median(cosine_max):.3f}')
axes[0, 1].set_title("Cosine — Max similarity to val set")
axes[0, 1].set_xlabel("Max cosine similarity")
axes[0, 1].legend()

# Tanimoto plots
axes[1, 0].hist(tanimoto_mean, bins=100, color='darkorange', edgecolor='none')
axes[1, 0].axvline(np.median(tanimoto_mean), color='k', linestyle='--', label=f'median={np.median(tanimoto_mean):.3f}')
axes[1, 0].set_title("Tanimoto — Mean similarity to val set")
axes[1, 0].set_xlabel("Avg Tanimoto similarity")
axes[1, 0].legend()

axes[1, 1].hist(tanimoto_max, bins=100, color='darkorange', edgecolor='none')
axes[1, 1].axvline(np.median(tanimoto_max), color='k', linestyle='--', label=f'median={np.median(tanimoto_max):.3f}')
axes[1, 1].set_title("Tanimoto — Max similarity to val set")
axes[1, 1].set_xlabel("Max Tanimoto similarity")
axes[1, 1].legend()

for ax in axes.flat:
    ax.set_ylabel("# train compounds")

plt.suptitle("Train vs Val Similarity (MACCS + ECFP + RDKit concat)", fontsize=14)
plt.tight_layout()
plt.savefig("train_val_similarity.png", dpi=150, bbox_inches='tight')
plt.show()

# UMAP of All Compounds (Tanimoto kNN, MACCS+ECFP+RDKit)

In [5]:
from rdkit.Chem.Scaffolds import MurckoScaffold

# Compute Murcko scaffold for every compound in all_metas
scaffolds = []
for meta in all_metas:
    mol = Chem.MolFromSmiles(meta.smiles)
    if mol is not None:
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    else:
        scaf = None
    scaffolds.append(scaf)

print(f"Computed {len(scaffolds)} scaffolds")
print(f"Unique scaffolds: {len(set(s for s in scaffolds if s))}")
print(f"Failed: {scaffolds.count(None)}")

Computed 34700 scaffolds
Unique scaffolds: 21606
Failed: 0


In [10]:
# Build full fingerprint matrix for all compounds
all_fps = np.array([get_fp(m) for m in all_metas], dtype=np.float32)
print(f"All compounds FP matrix: {all_fps.shape}")

# Labels for coloring
split_labels  = ['train' if m.smiles in train_set else 'val' for m in all_metas]
active_labels = [int(m.is_active_at_12_50) for m in all_metas]

# Top-20 scaffolds by frequency for scaffold coloring
from collections import Counter
scaf_counts = Counter(s for s in scaffolds if s)
top20 = {s for s, _ in scaf_counts.most_common(20)}
scaf_labels = [s if s in top20 else 'other' for s in scaffolds]

print(f"Split:  train={split_labels.count('train')}, val={split_labels.count('val')}")
print(f"Active: active={sum(active_labels)}, inactive={len(active_labels)-sum(active_labels)}")

All compounds FP matrix: (34700, 4263)
Split:  train=29774, val=4926
Active: active=456, inactive=34244


In [12]:
import anndata
import scanpy as sc

# kNN graph with Tanimoto (= Jaccard for binary fps) and 15 neighbors
adata = anndata.AnnData(X=all_fps)
adata.obs['split']    = split_labels
adata.obs['active']   = [str(a) for a in active_labels]
adata.obs['scaffold'] = scaf_labels

sc.pp.neighbors(adata, n_neighbors=15, use_rep='X', metric='jaccard')
sc.tl.umap(adata)
print("UMAP done. Embedding shape:", adata.obsm['X_umap'].shape)

/home/ethan2/.GrowthCurve/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


UMAP done. Embedding shape: (34700, 2)


In [ ]:
umap_coords = adata.obsm['X_umap']
u1, u2 = umap_coords[:, 0], umap_coords[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
s = 2  # point size

# Panel 1: train vs val
colors_split = ['steelblue' if l == 'train' else 'tomato' for l in split_labels]
axes[0].scatter(u1, u2, c=colors_split, s=s, alpha=0.4, rasterized=True)
axes[0].set_title("Train / Val split")
for label, color in [('train', 'steelblue'), ('val', 'tomato')]:
    axes[0].scatter([], [], c=color, s=30, label=label)
axes[0].legend(markerscale=3, frameon=False)

# Panel 2: activity (active at any time/concentration pair)
active_labels = [1 if (meta.pivot_cls == 1).any().any() else 0 for meta in all_metas]
colors_act = ['gold' if a == 1 else 'slategray' for a in active_labels]
axes[1].scatter(u1, u2, c=colors_act, s=s, alpha=0.4, rasterized=True)
axes[1].set_title("Activity (at least one active time-concentration pair)")
for label, color in [('active', 'gold'), ('inactive', 'slategray')]:
    axes[1].scatter([], [], c=color, s=30, label=label)
axes[1].legend(markerscale=3, frameon=False)

for ax in axes:
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("UMAP — all compounds, Tanimoto kNN (k=15), MACCS+ECFP+RDKit", fontsize=13)
plt.tight_layout()
plt.savefig("umap_all_compounds.png", dpi=150, bbox_inches='tight')
plt.show()